## Learning Objectives  
1. Learn the difference between genetic and genic variance.  
2. Calculate genic variance from scratch.  
3. Look at what happens to covariance between traits and loci due to selection.  
4. Assess the importance of loss of genic variance versus the generation of
linkage disequilibrium in decreasing the genetic variance.  
5. Look at effect of selection on allele frequencies.  
  

### Script setup  
Install packages

In [ ]:
#Loading libraries
req_packages<-c("AlphaSimR", "tidyverse")

for(i in c(1:length(req_packages))){
  if (!require(req_packages[i], character.only = TRUE)){
   install.packages(req_packages[i])
  }
}

#### Function to calculate genetic and genic variances and return a vec of them.

In [ ]:
getQTLEff <- function(SP){
  getOneTrait <- function(trait){
    return(trait@addEff)
  }
  return(sapply(SP$traits, getOneTrait))
}

#### Calculate genetic and genic variances from a population
Assuming this is a diploid

In [ ]:
calcGenGenicVars <- function(pop, SP){
  traitEff <- getQTLEff(SP)
  nTraits <- ncol(traitEff)
  popVarA <- AlphaSimR::varA(pop, simParam=SP)
  geneticVarCov <- c(popVarA[upper.tri(popVarA, diag=T)])
  # get genic covariances
  genicVarCov <- NULL
  qtlScores <- AlphaSimR::pullQtlGeno(pop, simParam=SP)
  qtlFreq <- colMeans(qtlScores) / 2
  qtlVar <- qtlFreq * (1 - qtlFreq)
  for (t1 in 1:nTraits){
    for (t2 in t1:nTraits){
      traitCrossProd <- traitEff[,t1]*traitEff[,t2]
      genicCov <- 2*sum(traitCrossProd*qtlVar)
      genicVarCov <- c(genicVarCov, genicCov)
    }
  }
  return(c(geneticVarCov, genicVarCov))
}

#### Select and calculate genetic and genic (co)variances
Breeding population stays at constant size  
If selIndex == NULL then all traits are weighted evenly  
If errVar == NULL then the error variance is 1 for all traits  

In [ ]:
selectCalcVar <- function(breedPop, selectFrac=0.2, selIndex=NULL,
                         errVar=NULL, nCyc=10, SP){
  nTraits <- length(SP$traits)
  popSize <- AlphaSimR::nInd(breedPop)
  if (is.null(selIndex)) selIndex <- rep(1, nTraits)
  if (is.null(errVar)) errVar <- rep(1, nTraits)

  popMean <- AlphaSimR::meanG(breedPop)
  genGenicVarCov <- c(0, popMean, calcGenGenicVars(breedPop, SP))
  for (cyc in 1:nCyc){
    # Phenotype the breeding population
    breedPop <- AlphaSimR::setPheno(breedPop, varE=errVar, simParam=SP)

    # Select parents for next generation
    nToSelect <- round(popSize * selectFrac, 0)
    phenos <- AlphaSimR::pheno(breedPop)
    index <- phenos %*% selIndex
    keep <-  index |> order(decreasing=T)
    keep <- keep[1:nToSelect]
    selected <- breedPop[keep]

    # Create new population by random mating
    # nCrosses: how many crosses to make
    # nProgeny: number of progeny per cross
    breedPop <- AlphaSimR::randCross(selected, nCrosses=popSize, nProgeny=1, simParam=SP)
    popMean <- AlphaSimR::meanG(breedPop)
    genGenicVarCov <- rbind(genGenicVarCov,
                            c(cyc, popMean, calcGenGenicVars(breedPop, SP)))
  }
  return(genGenicVarCov)
}


### Set up the founder and genome parameters  

In [ ]:
nFounders <- 1000
nBreedPop <- 200
nChr <- 7
repeatsPerFounder <- 10

### Run repeated selection programs: that there is a lot of noise with these effects

In [ ]:
runSelectionOnNQTL <- function(nQTL=10, nRepeats=20){
  nSegSites <- nQTL
  allRes <- NULL
  for (i in 1:nRepeats){
    if (i %% repeatsPerFounder == 1){
      cat(".")
      founderHaps <- AlphaSimR::runMacs(nInd=nFounders, nChr=nChr,
                                        segSites=nSegSites)
      SP <- AlphaSimR::SimParam$new(founderHaps)

      # `restrSegSites` prevents SNP from also being QTL
      # SP$restrSegSites(minQtlPerChr = nQTL, minSnpPerChr = nSNP, overlap = FALSE)
      # Specify parameters for two traits
      # The trait means, the additive variances and
      # the genetics correlations: 1 on the diagonal
      traitMeans <- c(0, 0)
      addVar <- c(1, 1)
      addCor <- matrix(c(1, 0, 0, 1), nrow=2) # Here, set zero genetic correlation
      SP$addTraitA(nQtlPerChr=nQTL, mean=traitMeans, var=addVar, corA=addCor)

      founders <- AlphaSimR::newPop(founderHaps, simParam=SP)
    }

    # Create a new population of founders
    # Create a new population of founders
    breedPop <- founders[sample(nFounders, nBreedPop)]
    res <- selectCalcVar(breedPop, selectFrac=0.2, selIndex=NULL,
                         errVar=NULL, nCyc=10, SP)
    allRes <- rbind(allRes, cbind(i, res))
  }
  rownames(allRes) <- NULL
  colnames(allRes) <- c("rep", "cycle", "popMeanT1", "popMeanT2",
                        "geneticVarT1", "geneticCov", "geneticVarT2",
                        "genicVarT1", "genicCov", "genicVarT2")
  return(allRes)
}

In [ ]:
meansAndPlot <- function(allRes, nQTL){
  meanByCyc <- as_tibble(allRes) |> group_by(cycle) |>
    dplyr::summarise(across(everything(), \(x) mean(x, na.rm = TRUE)))

  ggplot2::ggplot(meanByCyc, aes(x = genicCov, y = geneticCov, color = cycle)) +
    geom_point(size = 3) +  # Adjust point size as needed
    scale_color_gradient(low = "red", high = "blue") +  # Color gradient from red to blue
    theme_minimal() +  # Use a minimal theme
    labs(color = "Cycle", title=paste("nQTL = ", nQTL))
}

In [ ]:
allRes10 <- runSelectionOnNQTL(nQTL=10, nRepeats=100)
meansAndPlot(allRes10, nQTL=10)

In [ ]:
allRes200 <- runSelectionOnNQTL(nQTL=200, nRepeats=100)
meansAndPlot(allRes200, nQTL=200)

## Homework  
The lab today looked at the covariance between two traits and how it was
affected by LD between loci primarily affecting different traits and by fixation
of alleles that strongly affected both traits.  For the homework, I want you to
look at the variance of one trait and consider the same forces: LD between loci
affecting the trait and fixation of alleles that strongly affect the trait.

### Homework grading  

1. 5 points for a figure comparing genic and genetic variance for a trait
architecture with few QTL, along with some interpretation of what you see in the
figure.  
2. 5 points for a figure comparing genic and genetic variance for a trait
architecture with many QTL, along with some interpretation of what you see in
the figure.  
